In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import json
from utils import plot_timeseries_with_metadata, create_bounding_boxes, save_yolo_labels, visualize_bounding_boxes
from tqdm import tqdm

In [2]:
class PeakDetectPatient:
    def __init__(self, patient_id="03", root_path="NAVA_original/000_CNN_PeakDetectionData/"):
        self.patient_id = patient_id
        self.root_path = root_path
        self.all_os_paths = self.collect_os_paths_of_patient_data()
        self.peak_review_dict = self.get_peak_review_data()
        self.edi_sample_data_dict = self.get_edi_sample_data()
        self.edi_flow_pressure_data_dict = self.get_edi_flow_pressure_data()

        # self.check_missingness()
    
    def find_data_start_row(self, os_path, threshold=0.5):
        """
        Find the first row in an Excel file where data actually starts.
        Reads the file and finds the first row where at least 'threshold' fraction
        of columns have non-null values.
        """
        # Read the file without skipping rows to inspect all rows
        df_temp = pd.read_excel(os_path, header=None)
        
        # Find the first row where at least 'threshold' of columns are non-null
        # This row is likely the header or first data row
        for idx in range(len(df_temp)):
            non_null_ratio = df_temp.iloc[idx].notna().sum() / len(df_temp.columns)
            if non_null_ratio >= threshold:
                return idx
        
        # If no row meets the threshold, return 0 (start from beginning)
        return 0

    def collect_os_paths_of_patient_data(self):
        # go through the root path and collect os paths that starts with "edi-{patient_id}-"
        all_os_paths = []
        print(f"collecting xlsx files that starts with 'edi-{self.patient_id}-'")
        for root, dirs, files in os.walk(self.root_path):
            for file in files:
                if file.startswith(f"edi-{self.patient_id}"):
                    all_os_paths.append(os.path.join(root, file))
                if file.startswith(f"{self.patient_id}_raw"):
                    all_os_paths.append(os.path.join(root, file))
        all_os_paths.sort()
        print(f"Found {len(all_os_paths)} files for patient {self.patient_id}")
        print(all_os_paths)
        return all_os_paths

    def get_peak_review_data(self):
        # get the peak review data from the os path
        # find file that is inside Edi_peak_review folder
        peak_review_data_dict = {}
        for os_path in self.all_os_paths:
            if "Edi_peak_review" in os_path:
                st_peak_review_data = pd.read_excel(os_path, index_col=0, sheet_name="ST")
                bbb_peak_review_data = pd.read_excel(os_path, index_col=0, sheet_name="BbB")
                print(f"reading {os_path}")
                print(f"servotracker_peak_review_data shape: {st_peak_review_data.shape}")
                print(f"BbB_peak_review_data shape: {bbb_peak_review_data.shape}")
                peak_review_data_dict['ST'] = st_peak_review_data
                peak_review_data_dict['BbB'] = bbb_peak_review_data
                print("="*100)
                return peak_review_data_dict

    def get_edi_sample_data(self):
        edi_sample_data_dict = {}
        count = 1 # count=1 because file name starts with 1
        for os_path in self.all_os_paths:
            if "Edi_sample" in os_path:
                print(f"reading {os_path}")
                edi_sample_data_dict[count] = pd.read_excel(os_path, index_col=0)
                print(f"edi_sample_data_dict[{count}] shape: {edi_sample_data_dict[count].shape}")
                count += 1
        print("="*100)
        return edi_sample_data_dict
    
    def get_edi_flow_pressure_data(self):
        edi_flow_pressure_data_dict = {}
        count = 1 # count=1 because file name starts with 1
        for os_path in self.all_os_paths:
            if "Edi-flow_pressure_sample" in os_path:
                print(f"reading {os_path}")
                # Find the first row with actual data (no missing values)
                start_row = self.find_data_start_row(os_path)
                print(f"  Data starts at row {start_row} (skipping {start_row} metadata rows)")
                
                # Read the file, skipping metadata rows
                # The first non-skipped row will be treated as header by pandas
                if start_row > 0:
                    df = pd.read_excel(os_path, skiprows=range(start_row))
                else:
                    df = pd.read_excel(os_path)
                
                edi_flow_pressure_data_dict[count] = df
                print(f"edi_flow_pressure_data_dict[{count}] shape: {edi_flow_pressure_data_dict[count].shape}")
                count += 1
        print("="*100)
        return edi_flow_pressure_data_dict
        
    def check_missingness(self):# check if there is missingness in the peak_review_dict['ST']
        print(f"Missingness in ST: {self.peak_review_dict['ST'].isnull().sum()}")
        print(f"Missingness in BbB: {self.peak_review_dict['BbB'].isnull().sum()}")

        for key in self.edi_sample_data_dict.keys():
            print(f"Missingness in {key}: {self.edi_sample_data_dict[key].isnull().sum()}")
        for key in self.edi_flow_pressure_data_dict.keys():
            print(f"Missingness in {key}: {self.edi_flow_pressure_data_dict[key].isnull().sum()}")


In [3]:
root_path = "../../NAVA_original/000_CNN_PeakDetectionData/"
SAVE_DIR = "../EdiObjectDetectionData/"

WINDOW_SIZE = 2000
OVERLAP_SIZE = 0 # if 0 => No overlap. 
STRIDE = WINDOW_SIZE - OVERLAP_SIZE

FIGSIZE = (12, 4)
DPI = 100
PADDING = 3 # padding for the y-axis. 

NUM_WINDOWS_FOR_VISUALIZATION = 0 # Only save visualization for first 5 windows

# Class mapping for YOLO
class_mapping = {
    'brb_peak': 0,  # BbB peaks
}

# Bounding box parameters
BOX_WIDTH_RATIO = 0.01   # 1% of image width
BOX_HEIGHT_RATIO = 0.05  # 5% of image height

for patient_id in ["03", "04", "05"]:
    patient_data = PeakDetectPatient(patient_id, root_path)
    for edi_sample_data_dict_key in [1]: # [1, 2, 3, 4]
        total_length = len(patient_data.edi_sample_data_dict[edi_sample_data_dict_key])
        print(f"total length of edi_sample_data_dict[{edi_sample_data_dict_key}]: {total_length}")

        # Calculate number of windows
        num_windows = (total_length - WINDOW_SIZE) // STRIDE + 1
        print(f"Number of windows to process: {num_windows}")
        # Statistics tracking
        total_images = 0
        total_boxes = 0
        windows_with_peaks = 0

        # Iterate over sliding windows
        for window_idx in tqdm(range(num_windows), desc="Processing windows"):
            # Calculate window boundaries
            start_idx = window_idx * STRIDE
            end_idx = start_idx + WINDOW_SIZE
            
            # Ensure we don't go past the end
            if end_idx > total_length:
                break
            
            print(f"\n{'='*60}")
            print(f"Processing window {window_idx + 1}/{num_windows}")
            print(f"  Indices: {start_idx} to {end_idx}")
            
            # Extract timestamps for this window
            start_timestamp = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[start_idx]
            end_timestamp = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[end_idx - 1]
            
            # Find peaks within this window
            isin_timestamp_bbb = patient_data.peak_review_dict["BbB"]['Timestamp'].isin(
                patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[start_idx:end_idx]
            )
            subset_peak_review_df_bbb = patient_data.peak_review_dict["BbB"][isin_timestamp_bbb]
            
            # Extract EDI data and timestamps for this window
            edi_data = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Edi'].iloc[start_idx:end_idx]
            timestamps = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[start_idx:end_idx]
            
            # Prepare peaks data for this window
            peaks_data = []
            for idx, row in subset_peak_review_df_bbb.iterrows():
                peaks_data.append((row['Timestamp'], row['Edipeak'], "brb_peak"))
            
            print(f"  Number of peaks in this window: {len(peaks_data)}")
            


            # Image Folder
            save_image_path = f"{SAVE_DIR}/{patient_id}/images"
            label_path = f"{SAVE_DIR}/{patient_id}/labels"
            os.makedirs(save_image_path, exist_ok=True)
            os.makedirs(label_path, exist_ok=True)

            save_image_path_file_name = f"{save_image_path}/p{patient_id}_file_{edi_sample_data_dict_key}_window_{window_idx:04d}.png"
            save_label_path_file_name = f"{label_path}/p{patient_id}_file_{edi_sample_data_dict_key}_window_{window_idx:04d}.txt"

            # Step 1: Plot and save image
            try:

                metadata = plot_timeseries_with_metadata(
                    data=edi_data,
                    timestamps=timestamps,
                    save_img_path=save_image_path_file_name,
                    save_label_path=save_label_path_file_name,
                    figsize=FIGSIZE,
                    dpi=DPI,
                    padding=PADDING
                )
                
                # Step 2: Create bounding boxes from peaks
                boxes = create_bounding_boxes(
                    peaks_data=peaks_data,
                    timestamps=timestamps,
                    data=edi_data,
                    metadata=metadata,
                    box_width_ratio=BOX_WIDTH_RATIO,
                    box_height_ratio=BOX_HEIGHT_RATIO
                )
                
                # Step 3: Save YOLO format labels
                save_yolo_labels(boxes, save_label_path_file_name, class_mapping)
                
                # Step 4: Visualize for verification (optional - only for first few windows)
                if window_idx < NUM_WINDOWS_FOR_VISUALIZATION:  # Only save visualization for first 5 windows
                    viz_path = save_image_path_file_name.replace('.png', '_visualization.png')
                    visualize_bounding_boxes(save_image_path_file_name, boxes, viz_path)
                    print(f"  Visualization saved: {viz_path}")
                
                # Update statistics
                total_images += 1
                total_boxes += len(boxes)
                if len(boxes) > 0:
                    windows_with_peaks += 1
                
                print(f"  ✓ Saved: {save_image_path_file_name}")
                print(f"  ✓ Created {len(boxes)} bounding boxes")
                
            except Exception as e:
                print(f"  ✗ Error processing window {window_idx}: {str(e)}")
                continue


collecting xlsx files that starts with 'edi-03-'
Found 9 files for patient 03
['../../NAVA_original/000_CNN_PeakDetectionData/Edi-flow_pressure_sample/03/03_raw-1.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi-flow_pressure_sample/03/03_raw-2.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi-flow_pressure_sample/03/03_raw-3.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi-flow_pressure_sample/03/03_raw-4.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi_peak_review/edi-03_ST vs. BbB.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi_sample/03/edi-03-raw-1.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi_sample/03/edi-03-raw-2.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi_sample/03/edi-03-raw-3.xlsx', '../../NAVA_original/000_CNN_PeakDetectionData/Edi_sample/03/edi-03-raw-4.xlsx']
reading ../../NAVA_original/000_CNN_PeakDetectionData/Edi_peak_review/edi-03_ST vs. BbB.xlsx
servotracker_peak_review_data shape: (4772, 5)
BbB_p

Processing windows:   4%|▍         | 2/50 [00:00<00:03, 13.84it/s]


Processing window 1/50
  Indices: 0 to 2000
  Number of peaks in this window: 7
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//03/images/p03_file_1_window_0000.png
  ✓ Created 7 bounding boxes

Processing window 2/50
  Indices: 2000 to 4000
  Number of peaks in this window: 8
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//03/images/p03_file_1_window_0001.png
  ✓ Created 8 bounding boxes

Processing window 3/50
  Indices: 4000 to 6000
  Number of peaks in this window: 9


Processing windows:  12%|█▏        | 6/50 [00:00<00:02, 15.25it/s]

Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//03/images/p03_file_1_window_0002.png
  ✓ Created 9 bounding boxes

Processing window 4/50
  Indices: 6000 to 8000
  Number of peaks in this window: 9
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//03/images/p03_file_1_window_0003.png
  ✓ Created 9 bounding boxes

Processing window 5/50
  Indices: 8000 to 10000
  Number of peaks in this window: 11
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//03/images/p03_file_1_window_0004.png
  ✓ Created 11 bounding boxes

Processing window 6/50
  Indices: 10000 to 12000
  Number of peaks in this window: 11
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//03/images/p03_file_1_window_0005.png
  ✓ Created 11 bounding boxes

Processing window 7/50
  Indices: 12000 to 14000
  Number of peaks in this window: 7


...

  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0040.png
  ✓ Created 15 bounding boxes

Processing window 42/50
  Indices: 82000 to 84000
  Number of peaks in this window: 14
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0041.png
  ✓ Created 14 bounding boxes

Processing window 43/50
  Indices: 84000 to 86000
  Number of peaks in this window: 12
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0042.png
  ✓ Created 12 bounding boxes

Processing window 44/50
  Indices: 86000 to 88000
  Number of peaks in this window: 13
Actual image size: 1200x399 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0043.png
  ✓ Created 13 bounding boxes

Processing window 45/50
  Indices: 88000 to 90000
  Number of peaks in this window: 13


Processing windows:  94%|█████████▍| 47/50 [00:03<00:00, 14.14it/s]

Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0044.png
  ✓ Created 13 bounding boxes

Processing window 46/50
  Indices: 90000 to 92000
  Number of peaks in this window: 11
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0045.png
  ✓ Created 11 bounding boxes

Processing window 47/50
  Indices: 92000 to 94000
  Number of peaks in this window: 15
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0046.png
  ✓ Created 15 bounding boxes

Processing window 48/50
  Indices: 94000 to 96000
  Number of peaks in this window: 16
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0047.png
  ✓ Created 16 bounding boxes

Processing window 49/50
  Indices: 96000 to 98000
  Number of peaks in this window: 13


Processing windows: 100%|██████████| 50/50 [00:03<00:00, 14.18it/s]

Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0048.png
  ✓ Created 13 bounding boxes

Processing window 50/50
  Indices: 98000 to 100000
  Number of peaks in this window: 15
Actual image size: 1200x400 pixels
  ✓ Saved: ../EdiObjectDetectionData//05/images/p05_file_1_window_0049.png
  ✓ Created 15 bounding boxes


In [4]:


# ============================================================================
# MAIN WORKFLOW
# ============================================================================

# Your existing code
start_idx_of_edi = 5000 
end_idx_of_edi = 7000


start_timestamp_of_edi = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[start_idx_of_edi]
end_timestamp_of_edi = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[end_idx_of_edi]

time_diff = end_timestamp_of_edi - start_timestamp_of_edi 
print(f"Time difference: {time_diff}")

isin_timestamp_bbb = patient_data.peak_review_dict["BbB"]['Timestamp'].isin(
    patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[start_idx_of_edi:end_idx_of_edi])
subset_peak_review_df_bbb = patient_data.peak_review_dict["BbB"][isin_timestamp_bbb]

# Extract data
edi_data = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Edi'].iloc[start_idx_of_edi:end_idx_of_edi]
timestamps = patient_data.edi_sample_data_dict[edi_sample_data_dict_key]['Timestamp'].iloc[start_idx_of_edi:end_idx_of_edi]

# Prepare peaks data (ground truth) - NO y_buffer needed!
peaks_data = []
for idx, row in subset_peak_review_df_bbb.iterrows():
    peaks_data.append((row['Timestamp'], row['Edipeak'], "brb_peak"))

print(f"Number of peaks: {len(peaks_data)}")

# Step 1: Plot and save image (WITHOUT peaks drawn on it)
save_path = f'patient_{patient_id}_file_{edi_sample_data_dict_key}_idx_{start_idx_of_edi}.png'
metadata = plot_timeseries_with_metadata(
    data=edi_data,
    timestamps=timestamps,
    save_path=save_path,
    figsize=(12, 4),
    dpi=100,
    padding=2  # Absolute padding value
)

print(f"Image saved: {save_path}")
print(f"Metadata saved: {save_path.replace('.png', '_metadata.json')}")

# Step 2: Create bounding boxes from peaks
boxes = create_bounding_boxes(
    peaks_data=peaks_data,
    timestamps=timestamps,
    data=edi_data,  # Pass the data array
    metadata=metadata,
    box_width_ratio=0.01,   # 2% of image width
    box_height_ratio=0.05    # 10% of image height
)

print(f"Created {len(boxes)} bounding boxes")

# Step 3: Save YOLO format labels
label_path = save_path.replace('.png', '.txt')
class_mapping = {
    'brb_peak': 0,    # BbB peaks
}

save_yolo_labels(boxes, label_path, class_mapping)
print(f"YOLO labels saved: {label_path}")

# Step 4: Visualize for verification
viz_path = save_path.replace('.png', '_visualization.png')
visualize_bounding_boxes(save_path, boxes, viz_path)
print(f"Visualization saved: {viz_path}")


Time difference: 20000
Number of peaks: 9


TypeError: plot_timeseries_with_metadata() got an unexpected keyword argument 'save_path'

In [ ]:
timestamps

5000    1050020
5001    1050030
5002    1050040
5003    1050050
5004    1050060
         ...   
6995    1069970
6996    1069980
6997    1069990
6998    1070000
6999    1070010
Name: Timestamp, Length: 2000, dtype: int64

In [ ]:
metadata

{'image_path': 'patient_your_patient_id_file_2_idx_5000.png',
 'image_width': 1200,
 'image_height': 400,
 'timestamp_min': '1050020',
 'timestamp_max': '1070010',
 'timestamp_range': 19990.0,
 'data_min': 0.24,
 'data_max': 23.38,
 'y_min_plot': 0.13999999999999999,
 'y_max_plot': 23.48,
 'y_range_plot': 23.34,
 'num_datapoints': 2000,
 'figsize': (12, 4),
 'dpi': 100}

In [ ]:
peaks_data

[(np.float64(1052010.0), np.float64(6.17), 'brb_peak'),
 (np.float64(1054090.0), np.float64(10.87), 'brb_peak'),
 (np.float64(1056160.0), np.float64(9.5), 'brb_peak'),
 (np.float64(1058550.0), np.float64(5.46), 'brb_peak'),
 (np.float64(1060380.0), np.float64(4.55), 'brb_peak'),
 (np.float64(1061120.0), np.float64(9.13), 'brb_peak'),
 (np.float64(1062540.0), np.float64(14.73), 'brb_peak'),
 (np.float64(1063770.0), np.float64(23.38), 'brb_peak'),
 (np.float64(1065680.0), np.float64(20.19), 'brb_peak')]

In [ ]:
for sample_data_dict_key in [1, 2, 3, 4]:
    sample_data_df = patient_data.edi_sample_data_dict[sample_data_dict_key]
    # print(sample_data_df.head())
    # print(sample_data_df.tail())
    # print(sample_data_df.shape)
    print(sample_data_df['Edi'].describe())

count    100000.000000
mean          5.415813
std           8.280348
min           0.000000
25%           0.930000
50%           2.220000
75%           6.430000
max         144.240000
Name: Edi, dtype: float64
count    100000.000000
mean          3.541019
std           3.825564
min           0.000000
25%           1.040000
50%           2.190000
75%           4.630000
max          38.480000
Name: Edi, dtype: float64
count    100000.000000
mean          4.135110
std           4.628672
min           0.000000
25%           1.130000
50%           2.460000
75%           5.520000
max          48.080000
Name: Edi, dtype: float64
count    100000.000000
mean          2.788276
std           3.094605
min           0.000000
25%           0.910000
50%           1.910000
75%           3.560000
max          53.620000
Name: Edi, dtype: float64


In [ ]:


def save_yolo_labels(boxes, save_path, class_mapping={'blue': 0, 'red': 1, 'orange': 2, 'green': 3}):
    """
    Save bounding boxes in YOLO format
    
    Args:
        boxes: list of box dicts from create_bounding_boxes
        save_path: path to save .txt file
        class_mapping: dict mapping label names to class IDs
    
    Format: <class_id> <x_center> <y_center> <width> <height> (all normalized 0-1)
    """
    with open(save_path, 'w') as f:
        for box in boxes:
            label = box['label']
            class_id = class_mapping.get(label, 0)  # Default to 0 if label not in mapping
            
            bbox = box['bbox_normalized']
            f.write(f"{class_id} {bbox['x_center']:.6f} {bbox['y_center']:.6f} "
                   f"{bbox['width']:.6f} {bbox['height']:.6f}\n")


def visualize_bounding_boxes(image_path, boxes, save_path=None):
    """
    Visualize the bounding boxes on the image for verification
    
    Args:
        image_path: path to the saved image
        boxes: list of box dicts from create_bounding_boxes
        save_path: optional path to save visualization
    """
    from PIL import Image
    import matplotlib.patches as patches
    
    img = Image.open(image_path)
    fig, ax = plt.subplots(1, figsize=(14, 5))
    ax.imshow(img)
    
    # Color mapping for visualization
    color_map = {'blue': 'blue', 'red': 'red', 'orange': 'orange', 'green': 'green'}
    
    for box in boxes:
        bbox_px = box['bbox_pixels']
        label = box['label']
        color = color_map.get(label, 'yellow')
        
        # Calculate corner position (top-left)
        x_min = bbox_px['x_center'] - bbox_px['width'] / 2
        y_min = bbox_px['y_center'] - bbox_px['height'] / 2
        
        # Draw rectangle
        rect = patches.Rectangle(
            (x_min, y_min), bbox_px['width'], bbox_px['height'],
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label
        ax.text(bbox_px['x_center'], y_min - 5, label, 
                color=color, fontsize=8, ha='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    # plt.title('Bounding Boxes Visualization (for verification only)')
    plt.axis('off')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()
